# 07 - Orphan features: which pathways do the tiers suggest, and where is the THT example?

**Question.** For MEANtools features that have **no PlantMetWiki compound** (95 of 138 features; 79 compounds after merging adducts and isotopes, with a gene link), can the evidence tiers suggest the pathway the compound belongs to, and where does the THT example (the caffeoyltyramine hypothesis, notebook 06) fall among the candidates?

**Reading guide.** The notebook only **reads saved results** (no network, no heavy computation): `pilots/pairedomics_overlap/results/orphan_*.csv`, `metabolite_side_benchmark.csv`, `metabolite_side_tht.json`. The scripts that make them are `orphan_features.py`, `orphan_features_v2.py` and `metabolite_side_benchmark.py` (same folder); the written summary is [`../orphan-features-usecase.md`](../orphan-features-usecase.md).

**The short answer**
1. **Gene tiers alone** give a long list for each compound (a median of about 20 pathways per compound) and are **not better than a shuffled null** at the level of the whole dataset.
2. **THT is found**, but not at the top: with gene evidence only, the tyramine-amide pathway of the THT compound is **rank 74 of 92**; with the mass tier it is **rank 7**; requiring **agreement of the gene and the mass side** narrows 92 candidates to **3**, and it is one of them.
3. **The metabolite side** (mass differences to pathway metabolites, and structure similarity when a structure hypothesis exists) carries real signal in a benchmark with known answers.
4. **Agreement of independent sides** is the method that finds THT-like hits; across the 71 orphan compounds with gene candidates, 17 have one, against 10.3 expected by chance (P = 0.002).

In [1]:
import json, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
pd.set_option('display.width', 220, 'display.max_colwidth', 60, 'display.max_rows', 60)
INK, MUTED, LIGHT = '#1f2933', '#6b7280', '#e5e7eb'
BLUE, ORANGE, GREEN, RED, GREY = '#2a6f97', '#d9822b', '#3f8f5a', '#b5473f', '#9aa3ad'
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK, 'text.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})
def save(fig, name): fig.savefig(f'figures/{name}.png', dpi=170, bbox_inches='tight')
RES = Path('../../pilots/pairedomics_overlap/results')
def best(c): return c.sort_values(['score', 'n_genes_pathway'], ascending=False)
v1 = pd.read_csv(RES / 'orphan_candidates.csv')                 # gene tiers: locus, UniProt, EC, ortholog
v1s = pd.read_csv(RES / 'orphan_candidates_seq.csv')            # + transferred EC (UniRef50)
v2 = pd.read_csv(RES / 'orphan_candidates_v2_seq_gen.csv')      # + specificity weighting, mass tier (generic + specific deltas), mass-defect gate
THT = ['M322T887', 'M962T828']                                  # the two orphan compounds that the THT genes reach (M322T887 = [M+Na]+ peak of the hypothesis)
print(f"gene tiers: {v1.compound.nunique()} compounds, {len(v1):,} compound-pathway candidates; + transferred EC: {v1s.compound.nunique()} compounds, {len(v1s):,}; + mass tier (v2): {v2.compound.nunique()} compounds, {len(v2):,}")

gene tiers: 69 compounds, 1,585 compound-pathway candidates; + transferred EC: 71 compounds, 2,070; + mass tier (v2): 71 compounds, 2,070


## 1. What the candidates look like

One row per orphan **compound** (adducts and isotopes merged), showing its best pathway. `score` is the noisy-OR of the evidence families, `tiers` the gene tiers that reached the pathway, `n_genes_pathway` how many of the compound's correlated genes reach it, `mass_delta` whether the compound's mass differs by a conjugation from a pathway metabolite, and `p_pathway` / `p_any` the chance that **random genes** do as well (for this pathway / for any pathway). Pathway names that are empty are PlantCyc pathways without a title in the snapshot.

In [2]:
b = best(v2).drop_duplicates('compound')
cols = ['compound', 'mz', 'rt_s', 'pathway', 'score', 'w_membership', 'w_sequence', 'mass_delta', 'mass_defect_ok', 'n_genes_link', 'n_genes_pathway', 'p_pathway', 'p_any']
print(f"{len(b)} orphan compounds with at least one candidate pathway; best candidate per compound (top 20 by score):")
b[cols].head(20).reset_index(drop=True)

71 orphan compounds with at least one candidate pathway; best candidate per compound (top 20 by score):


,compound,mz,rt_s,pathway,score,w_membership,w_sequence,mass_delta,mass_defect_ok,n_genes_link,n_genes_pathway,p_pathway,p_any
0,M251T775,250.5648,775,D-galactose degradation I (Leloir pathway),0.700,0.400,0.5,False,True,28,2,0.008,0.1459
1,M352T804,330.1331,805,"D-myo-inositol (1,4,5)-trisphosphate biosynthesis",0.700,0.400,0.5,False,True,30,2,0.011,0.3237
2,M264T1547,241.0858,1547,PRPP biosynthesis,0.700,0.400,0.5,False,True,65,1,0.012,0.6034
3,M266T597,265.5706,597,galactolipid biosynthesis I,0.700,0.400,0.5,False,True,30,1,0.011,0.1558
4,M266T805,265.5702,805,alkane oxidation,0.700,0.400,0.5,False,True,67,1,0.011,0.3047
5,M298T530,298.1071,530,trehalose biosynthesis I,0.700,0.400,0.5,False,True,22,1,0.015,0.2448
6,M323T1528,323.1466,1528,calmodulin regulated calcium transport,0.700,0.400,0.5,False,True,42,1,0.022,0.3307
7,M328T594,328.1178,594,galactolipid biosynthesis I,0.700,0.400,0.5,False,True,17,1,0.015,0.2158
8,M328T773,328.1180,773,glycine biosynthesis,0.700,0.400,0.5,False,True,15,1,0.003,0.1528
9,M328T849,328.1185,849,glycine biosynthesis,0.700,0.400,0.5,False,True,17,1,0.003,0.1958


**What the table says.** The scores are mostly 0.4 to 0.7 and rest on an EC or ortholog link of **one** correlated gene (`n_genes_pathway` is 1 for most). The `p_any` column is the honest summary: for almost no compound is the best pathway better than what random genes give (next cell).

In [3]:
bb = best(v1s).drop_duplicates('compound')
print('Is the best pathway of a compound better than random genes would give? (p_any = chance that random genes of the same number reach a pathway at least this good)')
print(f"  compounds: {len(bb)};  p_any < 0.05: {(bb.p_any < 0.05).sum()};  expected by chance: {0.05 * len(bb):.1f}")
print(f"  median candidate pathways per compound: {v1s.groupby('compound').pc.nunique().median():.0f}")
print()
print('All candidates of the compounds are in the CSV files: results/orphan_candidates*.csv  (', len(v2), 'rows in the last version )')

Is the best pathway of a compound better than random genes would give? (p_any = chance that random genes of the same number reach a pathway at least this good)
  compounds: 71;  p_any < 0.05: 1;  expected by chance: 3.6
  median candidate pathways per compound: 20

All candidates of the compounds are in the CSV files: results/orphan_candidates*.csv  ( 2070 rows in the last version )


## 2. Where the THT example is found

The two orphan compounds that the THT genes reach are `M322T887` (the [M+Na]+ peak of the caffeoyltyramine hypothesis, merged with `M300T887` and `M301T887`) and `M962T828`. The figure ranks **all pathways that have gene evidence** for `M322T887` and marks the *hydroxycinnamic acid tyramine amides biosynthesis* pathway.

In [4]:
def rank_of(c, comp, pat='tyramine'):
    d = best(c[c.compound == comp]).reset_index(drop=True)
    i = d.index[d.pathway.fillna('').str.contains(pat)].tolist()
    return d, (i[0] + 1 if i else None)
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4), sharey=True)
for ax, (c, ttl) in zip(axes, [(v1s, 'gene tiers only (locus, UniProt, EC, ortholog, transferred EC)'), (v2, '+ mass tier and specificity weighting')]):
    d, r = rank_of(c, 'M322T887')
    x = np.arange(1, len(d) + 1)
    md = d.mass_delta.values if 'mass_delta' in d else np.zeros(len(d), bool)
    ax.scatter(x[~md], d.score[~md], s=22, color=GREY, lw=0, label='gene evidence only')
    if md.any(): ax.scatter(x[md], d.score[md], s=46, color=ORANGE, lw=0, label='+ mass differs by a conjugation from a pathway metabolite')
    ax.scatter([r], [d.score[r - 1]], s=190, facecolors='none', edgecolors=BLUE, lw=2.2, zorder=4, label='hydroxycinnamic acid tyramine amides')
    ax.annotate(f'THT pathway: rank {r} of {len(d)}', (r, d.score[r - 1]), xytext=(r + 6, d.score[r - 1] + 0.2), arrowprops=dict(arrowstyle='-', color=BLUE), color=BLUE, fontsize=9, fontweight='bold')
    ax.set_title(ttl, loc='left', fontsize=9.5); ax.set_xlabel('pathways with gene evidence for the compound, best first'); ax.set_ylim(0, 0.85)
axes[0].set_ylabel('score of the compound-pathway pair'); axes[1].legend(loc='upper right', frameon=False, fontsize=8)
fig.suptitle('The THT compound M322T887: the right pathway is in the list, but far from the top without the metabolite side', x=0.01, ha='left', fontsize=10.5, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.94)); save(fig, 'orphan_tht_rank'); plt.show()
for c, lab in ((v1s, 'gene tiers + transferred EC'), (v2, '+ mass tier')):
    for comp in THT:
        d, r = rank_of(c, comp); print(f'{lab:28s} {comp}: tyramine-amide pathway rank {r} of {len(d)}')

gene tiers + transferred EC  M322T887: tyramine-amide pathway rank 74 of 92
gene tiers + transferred EC  M962T828: tyramine-amide pathway rank 52 of 65
+ mass tier                  M322T887: tyramine-amide pathway rank 7 of 92
+ mass tier                  M962T828: tyramine-amide pathway rank 7 of 65


/var/folders/c8/yjbp_ly50415t5r79fxydc740000gn/T/ipykernel_65975/483647348.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(rect=(0, 0, 1, 0.94)); save(fig, 'orphan_tht_rank'); plt.show()


**Reading.** Without the metabolite side the right pathway is tied with about 50 others at score 0.5, below 41 pathways at 0.7 (EC plus ortholog hits on generic metabolism). The mass tier lifts it from rank 74 to rank 7 for the first compound. It does not put it first, because the gene weights dominate and several pathways have a higher or equal score.

## 3. Does the metabolite side carry signal? A benchmark with known answers

The THT example was chosen knowing the answer, so the metabolite side is tested on PlantMetWiki itself: each compound with a structure is **hidden** and its pathway is ranked from the metabolite side only (leave-one-out; 2,158 compounds, 962 pathways with at least 4 structures). Ties count as the midpoint.

In [5]:
bm = pd.read_csv(RES / 'metabolite_side_benchmark.csv')
bm['method'] = bm.method.str.replace('baseline: ', 'baseline: ').str.replace('mass-only: ', 'mass only: ')
show = ['baseline: random', 'baseline: pathway size', 'mass only: mass delta, specific conjugations', 'mass only: relative mass defect close to the pathway\'s',
        'mass only: mass delta, specific + generic (2H, O, CH2, H2O)', 'structure: RDKit path bits, top-3 mean', 'structure: Morgan r2 bits, top-3 mean', 'structure: atom pairs counts, top-3 mean', 'structure: Morgan r2 counts, top-3 mean']
t = bm.set_index('method').loc[show, ['top1', 'top5', 'top10', 'median_rank_fraction']]
fig, ax = plt.subplots(figsize=(10.5, 4.4)); y = np.arange(len(show))[::-1]
cols = [GREY if m.startswith('baseline') else (ORANGE if m.startswith('mass') else BLUE) for m in show]
ax.barh(y + 0.18, t.top5, height=0.34, color=cols, label='top-5 hit rate'); ax.barh(y - 0.18, t.top1, height=0.34, color=cols, alpha=0.55, label='top-1 hit rate')
for yy, a, b_ in zip(y, t.top5, t.top1): ax.text(a + 0.008, yy + 0.18, f'{a:.2f}', va='center', fontsize=8); ax.text(b_ + 0.008, yy - 0.18, f'{b_:.2f}', va='center', fontsize=8)
ax.set_yticks(y); ax.set_yticklabels([m.replace('mass only: ', '').replace('structure: ', 'structure: ') for m in show], fontsize=8.6); ax.set_xlim(0, 0.9)
ax.set_xlabel('share of hidden compounds whose true pathway is ranked in the top 1 (light) or top 5 (dark) of 962')
ax.legend(handles=[Patch(fc=GREY, label='baseline'), Patch(fc=ORANGE, label='mass only'), Patch(fc=BLUE, label='structure similarity')], loc='upper right', frameon=False, fontsize=8)
ax.set_title('Both the mass and the structure of a compound carry pathway information', loc='left', fontsize=10, fontweight='bold')
fig.tight_layout(); save(fig, 'orphan_benchmark'); plt.show()
t.round(3)

/var/folders/c8/yjbp_ly50415t5r79fxydc740000gn/T/ipykernel_65975/408605248.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, 'orphan_benchmark'); plt.show()


,top1,top5,top10,median_rank_fraction
method,,,,
baseline: random,0.005,0.020,0.034,0.292
baseline: pathway size,0.024,0.046,0.068,0.154
"mass only: mass delta, specific conjugations",0.047,0.138,0.172,0.501
mass only: relative mass defect close to the pathway's,0.119,0.239,0.278,0.055
"mass only: mass delta, specific + generic (2H, O, CH2, H2O)",0.218,0.517,0.620,0.005
"structure: RDKit path bits, top-3 mean",0.374,0.685,0.766,0.002
"structure: Morgan r2 bits, top-3 mean",0.413,0.750,0.834,0.002
"structure: atom pairs counts, top-3 mean",0.443,0.749,0.816,0.002
"structure: Morgan r2 counts, top-3 mean",0.452,0.770,0.839,0.002


**Reading.** Far above random and above the pathway-size baseline. **Structure similarity** is the strongest signal (count fingerprints, mean of the top 3 neighbours: top-5 77 %). **Mass alone** works when it uses the *generic* differences (2H, O, CH2, H2O) to other metabolites of the pathway (top-5 52 %); the specific conjugations alone are weak (14 %). The benchmark is **optimistic**: the hidden compounds usually have reaction partners in their own pathway.

## 4. The THT compound under each method, and a method that finds it

The tyramine-amide pathway (PWY-5474) of the THT compound, by the evidence that is used (rank 1 is best; `N` is the number of pathways in that ranking).

In [6]:
tj = json.load(open(RES / 'metabolite_side_tht.json'))['ranks']
d1, r1 = rank_of(v1s, 'M322T887'); d2, r2 = rank_of(v2, 'M322T887')
ag = d2[d2.mass_delta]; rag = int(ag.reset_index(drop=True).index[ag.reset_index(drop=True).pathway.fillna('').str.contains('tyramine')][0]) + 1
rows = [('gene tiers only (transferred EC, ortholog, EC ...)', 'gene', r1, len(d1)), ('gene tiers + mass tier (v2)', 'both', r2, len(d2)),
        ('agreement: gene-supported AND mass-delta pathways', 'both', rag, len(ag)),
        ('mass only: delta to pathway metabolites', 'metabolite', tj['mass-only: delta specific + generic'], 962),
        ('structure of the hypothesis: atom pairs (oracle)', 'metabolite', tj['structure oracle: atom pairs counts, top-3 mean'], 962),
        ('structure of the hypothesis: Morgan counts (oracle)', 'metabolite', tj['structure oracle: Morgan r2 counts, top-3 mean'], 962),
        ('structure of the hypothesis: Morgan bits (oracle)', 'metabolite', tj['structure oracle: Morgan r2 bits, top-3 mean'], 962)]
rk = pd.DataFrame(rows, columns=['evidence', 'side', 'rank', 'out of'])
fig, ax = plt.subplots(figsize=(10.5, 3.9)); y = np.arange(len(rk))[::-1]
cmap = {'gene': GREY, 'both': GREEN, 'metabolite': ORANGE}
ax.barh(y, rk['rank'] / rk['out of'], color=[cmap[s_] for s_ in rk.side], height=0.6)
for yy, r_, n_ in zip(y, rk['rank'], rk['out of']): ax.text(r_ / n_ + 0.01, yy, f'rank {r_:g} of {n_}', va='center', fontsize=8.6)
ax.set_yticks(y); ax.set_yticklabels(rk.evidence, fontsize=8.6); ax.set_xlim(0, 1.05); ax.set_xlabel('rank as a fraction of the pathways in the list (smaller is better)')
ax.legend(handles=[Patch(fc=GREY, label='gene side'), Patch(fc=ORANGE, label='metabolite side'), Patch(fc=GREEN, label='both sides')], loc='lower right', frameon=False, fontsize=8)
ax.set_title('Where the THT pathway is found: not singled out by genes; first by the metabolite side; one of 3 when both agree', loc='left', fontsize=9.6, fontweight='bold')
fig.tight_layout(); save(fig, 'orphan_tht_methods'); plt.show(); rk

/var/folders/c8/yjbp_ly50415t5r79fxydc740000gn/T/ipykernel_65975/2201319985.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, 'orphan_tht_methods'); plt.show(); rk


,evidence,side,rank,out of
0,"gene tiers only (transferred EC, ortholog, EC ...)",gene,74.0,92
1,gene tiers + mass tier (v2),both,7.0,92
2,agreement: gene-supported AND mass-delta pathways,both,2.0,3
3,mass only: delta to pathway metabolites,metabolite,1.0,962
4,structure of the hypothesis: atom pairs (oracle),metabolite,1.0,962
5,structure of the hypothesis: Morgan counts (oracle),metabolite,2.0,962
6,structure of the hypothesis: Morgan bits (oracle),metabolite,4.0,962


**Reading.** The gene side puts the right pathway among many equally supported ones; the **mass side** (differences to pathway metabolites: one OH above p-coumaroyltyramine, one CH2 below feruloyltyramine) and a **structure hypothesis** put it first among 962; the **agreement** of both leaves **3 of 92** candidates, including the right one (rank 2 of 3 by score: the three agreeing pathways are suberin monomers, the tyramine amides and Amaryllidacea alkaloids, with nearly equal scores). Structure ranks are for the top-3-mean aggregation (with the single best hit, Morgan bits give rank 13.5 and RDKit path 20). The oracle rows use the hand-written structure of the hypothesis and are not available for real orphans.

### The agreement method
1. A **gene-side** candidate list: pathways reached by a correlated gene through the locus, UniProt, EC, ortholog or transferred-EC tiers.
2. A **metabolite-side** list: pathways with a metabolite whose mass differs from the compound by a conjugation (acetyl, sugar, caffeoyl, ... and 2H, O, CH2, H2O), and when a structure hypothesis exists, the pathways of the most similar compounds (count fingerprints, top-3 mean).
3. The **intersection**, ranked by the combined score. It is tested against a null in which the gene annotation is shuffled.

In [7]:
g = best(v2)
a = g[g.mass_delta]
ab = a.drop_duplicates('compound')
bb2 = g.drop_duplicates('compound')
print(f"{len(bb2)} orphan compounds with gene candidates; {bb2.compound_agrees.sum()} have a pathway supported by both the gene tiers and a mass delta; expected {bb2.p_agree.sum():.1f} when the gene annotation is shuffled")
p = bb2.p_agree.values; sims = (np.random.default_rng(0).random((20000, len(p))) < p).sum(1)
print(f"P(at least {bb2.compound_agrees.sum()} by chance) = {(sims >= bb2.compound_agrees.sum()).mean():.3f}")
ag_tab = a.groupby('compound').agg(pathways=('pathway', lambda s: '; '.join(s.fillna('(no title)').head(3)) + (' ...' if len(s) > 3 else '')), n_pathways=('pathway', 'size'), best_score=('score', 'max'), mz=('mz', 'first'), p_agree=('p_agree', 'first')).sort_values('best_score', ascending=False)
ag_tab['THT'] = ag_tab.index.isin(THT)
ag_tab.round(3)

71 orphan compounds with gene candidates; 17 have a pathway supported by both the gene tiers and a mass delta; expected 10.3 when the gene annotation is shuffled
P(at least 17 by chance) = 0.002


,pathways,n_pathways,best_score,mz,p_agree,THT
compound,,,,,,
M530T485,secologanin and strictosidine biosynthesis,1,0.600,530.160,0.214,False
M342T713,secologanin and strictosidine biosynthesis,1,0.600,342.133,0.156,False
M190T888,"vindoline, vindorosine and vinblastine biosynthesis; sup...",56,0.600,190.051,0.972,False
M322T887,suberin monomers biosynthesis; hydroxycinnamic acid tyra...,3,0.460,300.122,0.388,True
M264T1547,tetrahydrofolate biosynthesis II; superpathway of purine...,8,0.455,241.086,0.655,False
M962T828,suberin monomers biosynthesis,1,0.400,962.339,0.362,True
M329T717,sesamin biosynthesis,1,0.388,328.118,0.086,False
M162T1011,superpathway of photosynthetic hydrogen production; glut...,35,0.380,161.539,1.000,False
M161T1547,glycerol degradation I; phospholipases; phospholipases (...,18,0.380,160.533,1.000,False


**Reading.** 17 compounds have an agreeing pathway, against 10.3 expected: a modest excess (about 7 compounds, P = 0.002), the first signal above chance in the orphan analysis. For a single compound the agreement is **not** significant (for the two THT compounds a random gene set agrees 36 to 39 % of the time), and the median compound still has 3 agreeing pathways (5 of 17 have exactly one).

## 5. What this does and does not show

- **Where THT is found.** Among the gene-supported pathways of `M322T887` (92) the tyramine-amide pathway is at rank 74 by genes alone, 7 with the mass tier, and one of 3 (rank 2 of 3) when gene and mass agree; by the mass side alone it is first of 962.
- **A method for THT-like hits:** agreement of independent sides, with the metabolite side from pathway-neighbour mass differences and, when available, a structure hypothesis.
- **Limits.** THT is the development case. The generic mass differences were reinstated after the benchmark and the THT rank (a forking path). The benchmark is leave-one-out inside PlantMetWiki and optimistic. The agreement is still a short list, not a single answer. There are **no structure candidates for real orphans** (the annotation table of the dataset is not mass-consistent), so the structure route is only demonstrated with the hypothesis. No MS/MS or standard supports the caffeoyltyramine identification (level 3-4). Weights, deltas, the gate margin and the 50 % sequence-cluster threshold are drafts.
- **Next.** Test the agreement method on the ipecac pilot, where the answers are known, and add a structure source by formula for real orphans.